# Śledzenie obiektów

<img src="https://i.imgur.com/wKXXFkQ.png" width="500">

## Wstęp
W erze cyfrowej, w obliczu rosnącej lawinowo ilości danych wideo, zdolność do ich automatycznego rozpoznawania i interpretowania staje się kluczowa w wielu dziedzinach – od bezpieczeństwa publicznego po autonomiczne pojazdy. Technologie oparte na głębokim uczeniu rewolucjonizują sposób, w jaki przetwarzamy informacje wizualne. Kluczowym wyzwaniem jest tu detekcja i śledzenie obiektów na filmach wideo.

Celem tego zadania jest opracowanie algorytmu, który będzie w stanie analizować sekwencje ruchów w grze "trzy kubki". Uczestnicy mają za zadanie określić końcową pozycję kubków po serii ruchów, korzystając z analizy statycznych obrazów z każdej klatki nagrania.

In [ ]:
######################### NIE ZMIENIAJ TEJ KOMÓRKI PODCZAS WYSYŁANIA ##########################

# Poniższe funkcje ułatwiają pracę z dostarczonymi danymi
# W kolejnych komórkach zobaczysz przykłady ich użycia
from utils.utils import get_level_info, get_video_data, display_video, download_and_replace_data

FINAL_EVALUATION_MODE = False
# W czasie sprawdzania Twojego rozwiązania, zmienimy tę wartość na True
# Wartość tej flagi M U S I zostać ustawiona na False w rozwiązaniu, które nam nadeślesz!

images, coordinates, _, path_to_images = get_video_data(level=3,video_id=0,dataset="example")
display_video(images,rescale=0.7,FINAL_EVALUATION_MODE=FINAL_EVALUATION_MODE)

## Zadanie 3: Zbuduj rozwiązanie od zera

W tym zadaniu podejdziesz do problemu identyfikacji obiektów na filmach wideo od podstaw. Poprzednie zadania wymagały użycia gotowych informacji o lokalizacji obiektów w klatkach. Tym razem będziesz musiał stworzyć algorytm, który będzie operować bezpośrednio na nieoznaczonych obrazach, co pozwoli na pełniejsze zrozumienie i opracowanie własnego systemu detekcji obiektów.

Napisz algorytm, który poradzi sobie ze zbiorem danych `level_3` bez podanych prostokątów ograniczających.

## Pliki zgłoszeniowe
Tylko ten notebook zawierający **kod** oraz **krótki raport** opisujący Twoje rozwiązanie (do 300 słów). Miejsce na raport znajdziesz na końcu tego notebooka.

## Ograniczenia
- Twoja funkcja powinna zwracać predykcje w maksymalnie 5 minut używając Google Colab bez GPU.

## Uwagi i wskazówki
- Testuj swoje rozwiązanie na zbiorze plików wideo `level_3`.
- **Skuteczność modelu**: przetestuj skuteczność modelu na zbiorze walidacyjnym używając dostarczonej przez nas funkcji **submission_script**, umieść ten wynik w raporcie.

## Ewaluacja
Pamiętaj, że podczas sprawdzania flaga `FINAL_EVALUATION_MODE` zostanie ustawiona na `True`. Za pomocą skryptu `validation_script.py` możesz upewnić się, że Twoje rozwiązanie zostanie prawidłowo wykonane na naszych serwerach oceniających.

Za to podzadanie możesz zdobyć pomiędzy 0 i 0.5 punktów. Zdobędziesz 0 punktów jeśli Twoje accuracy na zbiorze testowym będzie poniżej 30%. Jeśli będzie równe 100%, otrzymasz 0.5 punktu. Pomiędzy tymi wartościami, wynik rośnie liniowo z wartością metryki.

# Kod startowy

In [ ]:
# Poniższe biblioteki są wystarczające do wykonania wszystkich zadań
# Jeśli jednak chcesz użyć innych, sprawdź czy są dostępne na serwerze (requirements.txt)
import numpy as np
import os
import matplotlib.pyplot as plt
import torch
import IPython.display
import json
import PIL
import sklearn as sk
import gdown
import os

In [ ]:
# funkcja pomocnicza do ładowania danych
images, _, _, _ = get_video_data(level=3,video_id=0,dataset="example")

with open(os.path.join(os.getcwd(),'example_tracks','tracks_3_0.json'), 'r') as f:
    tracks = json.load(f)

for key in tracks.keys():
    tracks[key] = [tuple(el) for el in tracks[key]]

# funkcja pomocnicza do wyświetlania danych
display_video(images,
                tracks=tracks,
                rescale=0.7,
                FINAL_EVALUATION_MODE=FINAL_EVALUATION_MODE)

In [ ]:
# Pobieranie danych do podzadań 1, 2 i 3 (około ~646Mb), skrypt będzie wykonywał się parę minut
# Wystarczy, że pobierzesz dane tylko raz. Na serwerze sprawdzającym dane będą już pobrane,
# struktura plików będzie identyczna jak tutaj
if not FINAL_EVALUATION_MODE:
    download_and_replace_data()


In [ ]:
######################### NIE ZMIENIAJ TEJ KOMÓRKI ##########################

# funkcja pomocnicza do testowania algorytmu
# Zwróć uwagę na to że funkcja ta działa inaczej niż w poprzednich podzadaniach
# algorytm przyjmuje jako argument listę obrazków, a nie koordynaty
def submission_script(algorithm,level,verbose=False,dataset="valid"):
    num_videos, _ = get_level_info(level=level,dataset=dataset)
    correct = []
    exception_messages = set()
    for video_number in range(num_videos):
        images, _, target, _ = get_video_data(level=level,video_id=video_number,dataset=dataset)
        try:
            prediction = algorithm(images)
            if tuple(target) == tuple(prediction):
                correct.append(1)
            else:
                correct.append(0)
            if verbose:
                print(f"Video: animation_{str(video_number).zfill(4)}")
                print(f"Prediction: {prediction}")
                print(f"Target:     {target}")
                print(f"Score: {tuple(target) == tuple(prediction)}", end='\n\n')
        except Exception as e:
            correct.append(0)
            exception_messages.add(str(e))
    if verbose:
        print(f"Accuracy: {np.mean(correct)}")
        print(f"Correctness: {correct}")
    return np.sum(correct) / num_videos, correct, exception_messages

# Twoje rozwiązanie

In [ ]:
import numpy as np
from sklearn.cluster import KMeans
from scipy.optimize import linear_sum_assignment


def your_algorithm_task_3(images):
    def task_1(coordinates):
        frames = np.array(coordinates)
        frames = frames[:, :, np.array([1, 0])]
        start_sort = np.argsort(frames[0, :, 0])
        start_pos = np.arange(3)
        start_pos[start_sort] = np.arange(3)

        permutation = np.arange(3)

        for idx, frame in enumerate(frames[1:]):
          costs = np.zeros((3, 3))
          for i in range(3):
            for j in range(3):
              costs[i, j] = np.sum((frame[j] - frames[idx, i])**2)

          row_indices, col_indices = linear_sum_assignment(costs)

          permutation = col_indices[permutation]

        return start_pos[np.argsort(frames[-1, permutation, 0])]

    frames = np.array(images)

    boxes = []
    for frame in frames:
      mask = frame[:, :, 2] < 20
      coords = np.column_stack(np.where(mask))
      if boxes:
        kmeans = KMeans(3, n_init=1, init=boxes[-1])
      else:
        kmeans = KMeans(3, n_init='auto')
      kmeans.fit(coords)
      boxes.append(kmeans.cluster_centers_)

    return task_1(boxes)



In [ ]:
# Sprawdź jak działa Twój algorytm
accuracy, correctness, _ = submission_script(
    algorithm=your_algorithm_task_3,
    level=3,
    verbose=True,
    dataset="train")

In [ ]:
# zapisz swój raport do zmiennej poniżej, abyśmy mogli go później automatycznie odczytać sprawdzaczką
raport_3 = \
"""
Raport z zadania:
Dla każdej klatki tworze maskę, aby zaznaczyć pixele w których znajdują się kubki. Nastepnie dokonuję klasteryzacji na 3 klastry przez algorytm KMeans. Środki klastrów uznaję za środki kubków. Następnie wykonuję algorytm analogiczny do podzadania 1
"""